# Assignment 1

Deadline: 19.03.2026, 12:00 CET

- Gallo Alessandro , 25-732-140 , alessandro.gallo2@uzh.ch
- Venturi Matilde , 25-741-059 , matilde.venturi@uzh.ch

In [1]:
# Import standard libraries
import os
import sys
import time # To compute runtimes
from typing import Optional

# Import third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import local modules
project_root = os.path.dirname(os.path.dirname(os.getcwd()))   # Change this path if needed
src_path = os.path.join(project_root, 'qpmwp-course', 'src') #changed for Macbook
sys.path.append(project_root)
sys.path.append(src_path)
from estimation.covariance import Covariance, is_pos_def, make_pos_def
from estimation.expected_return import ExpectedReturn
from optimization.constraints import Constraints
from optimization.optimization import Optimization, Objective
from optimization.optimization_data import OptimizationData
from optimization.quadratic_program import QuadraticProgram, USABLE_SOLVERS
from helper_functions import simulate_correlated_gbm
from optimization import quadratic_program

## 1. Solver Horse Race

### 1.a)
(3 points)

Generate a synthetic dataset of dimension TxN, T=1000, N=50, and compute a vector of expected returns, q, and a covariance matrix, P, using classes ExpectedReturn and Covariance respectively.

In [2]:
# Set the dimensions
T = 1000       # Number of time steps
N = 50         # Number of assets
rnd_seed = 42  # Random seed for reproducibility

# Set random seed for reproducibility
np.random.seed(rnd_seed)



# 1. Generate a random mean vector and covariance matrix using a multivariate normal distribution

# We first generate a random dataset of returns, and then compute the mean vector and covariance matrix from that dataset.
# This ensures that the covariance matrix is based on actual data, which is more realistic for financial applications.
# Note: finanacial returns are not normally distributed, but this is just for demonstration purposes.
data = np.random.randn(T, N)   
#df = pd.DataFrame(data)

mu=np.mean(data,axis=0)
sigma = np.cov(data, rowvar=False)



# 2. Make sure the covariance matrix is positive definite.

# Check if the covariance matrix is positive definite using is_pos_def.
# If not, make it positive definite with make_pos_def. Cholesky decomposition is used in is_pos_def for the check.
# Note: A covariance matrix must be positive definite to be valid, as it represents the variance and covariance of asset returns.
# If the covariance matrix is not positive definite, it can lead to issues in optimization algorithms that rely on it, such as mean-variance optimization.
if not is_pos_def(sigma):
    print("Covariance matrix is not positive definite. Making it positive definite.")
    sigma = make_pos_def(sigma)



# 3. Generate correlated geometric Brownian motion paths and compute discrete returns

# Generate returns using simulate_correlated_gbm, which simulates correlated geometric Brownian motion paths from mu and sigma.
# Use simulated prices to compute discrete returns for expected return and covariance estimation.
prices = simulate_correlated_gbm(mu=mu, sigma=sigma, T=T, random_seed=None)
returns = prices.pct_change().dropna()



# 4. Compute the vector of expected returns from df using class ExpectedReturn
scalefactor = 1  # could be set to 252 (trading days) for annualized returns

expected_return = ExpectedReturn(
    method='geometric',
    scalefactor=scalefactor
)

q = expected_return.estimate(X=returns, inplace=False)


# 5. Compute the covariance matrix from df using class Covariance
covariance = Covariance(method="pearson")
P = covariance.estimate(X=returns, inplace=False)


# 6. Display the results
print("Vector of expected returns (q):")
print(q)

print("\nCovariance matrix (P):")
print(P)

Vector of expected returns (q):
Asset_1    -0.000425
Asset_2    -0.001279
Asset_3    -0.001210
Asset_4     0.000285
Asset_5    -0.001033
Asset_6     0.000211
Asset_7    -0.000808
Asset_8     0.000655
Asset_9    -0.001226
Asset_10   -0.002073
Asset_11    0.000321
Asset_12   -0.000574
Asset_13    0.000176
Asset_14   -0.000409
Asset_15    0.000274
Asset_16   -0.000498
Asset_17   -0.001069
Asset_18   -0.001996
Asset_19   -0.001117
Asset_20   -0.000353
Asset_21    0.000656
Asset_22   -0.000628
Asset_23   -0.000761
Asset_24   -0.001393
Asset_25    0.000497
Asset_26    0.000331
Asset_27   -0.001047
Asset_28    0.000610
Asset_29   -0.001153
Asset_30    0.000681
Asset_31   -0.001564
Asset_32   -0.000975
Asset_33    0.000343
Asset_34   -0.000689
Asset_35    0.000794
Asset_36    0.000242
Asset_37    0.000110
Asset_38    0.001863
Asset_39    0.000280
Asset_40   -0.001006
Asset_41   -0.001469
Asset_42   -0.000127
Asset_43   -0.001588
Asset_44   -0.000115
Asset_45    0.001424
Asset_46   -0.000873
As

### 1.b)
(3 points)

Instantiate a constraints object by injecting column names of the return series created in 1.a) as ids and add:
- a budget constaint (i.e., asset weights have to sum to one)
- lower bounds of 0.0 for all assets
- upper bounds of 0.2 for all assets
- group contraints such that the sum of the weights of the first 15 assets is <= 0.3, the sum of assets 16 to 45 is <= 0.4 and the sum of assets 41 to 50 is <= 0.5

In [3]:
# 1. Instantiate the Constraints class
constraints = Constraints(ids = returns.columns.tolist())



# 2. Add budget constraint
# Setting the budget constraint with rhs=1 and sense="=" ensures that the sum of the portfolio weights equals 1,
# which is a common requirement in portfolio optimization to represent a fully invested portfolio.
constraints.add_budget(rhs=1,sense="=")



# 3. Add box constraints (i.e., lower and upper bounds)
# Setting the box constraints with lower=0 and upper=0.2 ensures that each asset weight is between 0 and 0.2.
constraints.add_box(lower=0,upper=0.2)



# 4. Add linear constraints
# We create a G matrix where each row corresponds to a linear constraint and each column corresponds to an asset.
# The first constraint ensures that the sum of the weights of the first 15 assets is less than or equal to 0.3.
# The second constraint ensures that the sum of the weights of the next 30 assets (16 to 45) is less than or equal to 0.4.
# The third constraint ensures that the sum of the weights of the last 10 assets (40 to 50) is less than or equal to 0.5.
G = pd.DataFrame(0, index=range(3), columns=constraints.ids)
G.iloc[0, 0:15] = 1
G.iloc[1, 15:45] = 1
G.iloc[2, 40:50] = 1
h = pd.Series([0.3, 0.4, 0.5])
constraints.add_linear(G=G, sense="<=", rhs=h)



# 5. Display some columns of the G matrix to verify that the constraints have been set up correctly
# This will show the columns corresponding to the assets that are involved in the linear constraints.
constraints.linear['G'][['Asset_1', 'Asset_15', 'Asset_16', 'Asset_40', 'Asset_41', 'Asset_50']]

,Asset_1,Asset_15,Asset_16,Asset_40,Asset_41,Asset_50
0,1,1,0,0,0,0
1,0,0,1,1,1,0
2,0,0,0,0,1,1


### 1.c) 
(4 points)

Solve a Mean-Variance optimization problem (using coefficients P and q in the objective function) which satisfies the above defined constraints.
Repeat the task for all open-source solvers in qpsolvers that you could install and compare the results in terms of:

- runtime
- accuracy: value of the primal problem.
- reliability: are all constraints fulfilled? Extract primal residuals, dual residuals and duality gap.

Generate a DataFrame with the solvers as column names and the following row index: 'solution_found': bool, 'objective': float, 'primal_residual': float, 'dual_residual': float, 'duality_gap': float, 'runtime': float.

Put NA's for solvers where the optimization failed for some reason.




## **Exercise 1.c Technical Notes**

### Why are we converting vectors and matrices?

Optimization solvers usually expect inputs in a very strict numerical format.

In our code, some objects come from **pandas** (`Series`, `DataFrame`), while the solver expects mainly:

- **vectors** as `numpy.ndarray`
- **matrices** as `numpy.ndarray` or `scipy.sparse` matrices

That is why we convert:

- `q`, `h`, `b`, `lb`, `ub` into NumPy vectors
- `P`, `G`, `A` into NumPy matrices or sparse matrices

These conversions are important because pandas objects carry labels and indexing information, which is useful for analysis, but solver backends only want raw numerical arrays. If we pass pandas objects directly, many solvers raise type errors.

### Why do we sometimes use sparse matrices?

Some matrices, especially constraint matrices like `G` and `A`, contain many zeros. In that case, using a **sparse matrix** is more efficient because it stores only the nonzero entries.

This helps by:

- reducing memory usage
- speeding up computations
- matching the preferred input format of some solvers

### Why do we use `-q`?

Most quadratic programming solvers work in **minimization** form:

$$
\min \frac{1}{2} x^T P x + q^T x
$$

In portfolio optimization, we often want to **maximize return** while penalizing risk, for example:

$$
\max \mu^T x - \frac{\lambda}{2} x^T \Sigma x
$$

To give this problem to the solver, we rewrite it as a minimization:

$$
\min \frac{\lambda}{2} x^T \Sigma x - \mu^T x
$$

So if $q$ originally represents expected returns $\mu$, we pass $-q$ to the solver.  
This is not an arbitrary change: it is how we translate a maximization problem into the minimization form required by the solver.

### Summary

We do these conversions because:

1. the solver needs plain numerical arrays, not pandas objects  
2. sparse matrices are more efficient for large constraint systems  
3. $-q$ is used to convert a return-maximization problem into the solver’s minimization form

In [4]:
from scipy import sparse

# Helper fuctions for optimization
# We will use these function to make the parameters suitable for the optimization solvers, which typically require numpy arrays or sparse matrices.

# Check that the vector is in the correct format for optimization solvers.
def as_vec(x):
    if x is None:
        return None
    if isinstance(x, pd.Series):
        return x.to_numpy(dtype=np.float64)
    arr = np.asarray(x, dtype=np.float64)
    return arr.reshape(-1)

# Check that the matrix is in the correct format for optimization solvers.
def as_mat(x):
    if x is None:
        return None
    if sparse.issparse(x):
        return x.tocsc().astype(np.float64)
    if isinstance(x, pd.DataFrame):
        return x.to_numpy(dtype=np.float64)
    arr = np.asarray(x, dtype=np.float64)
    return np.atleast_2d(arr)

We install all open-source solvers listed in `USABLE_SOLVERS` from `optimization.quadratic_program.py`. These solvers are required to run the mean-variance optimization and compare their performance in terms of runtime, accuracy, and reliability. The installation ensures that each solver is available for benchmarking in the subsequent analysis.

In [5]:
!pip install qpsolvers cvxopt quadprog osqp daqp qpalm

In [6]:
# 0. Import qpsolvers and the QuadraticProgram class
from optimization import quadratic_program



# 1. Extract the constraints in the format required by the solver
GhAb = constraints.to_GhAb()

# 2. Define a dictionary to store the results in case a solver fails
result_on_fail =  {
    'solution_found': False,
    'objective': np.nan,
    'primal_residual': np.nan,
    'dual_residual' : np.nan,
    'duality_gap': np.nan,
    'runtime': np.nan,
    'error': None
}

# 3. Set risk aversion parameter and get all available solvers from qpsolvers
risk_aversion = 1
available_solvers = quadratic_program.USABLE_SOLVERS
results_dict = {}

# We will store also the weights generated by the different optimizers in a dictionary, which can be useful for the enxt exercise.
weights_dict = {}



# 4. Convert the problem data to the format required by the solvers
P_np  = as_mat(P)
P_np  = 0.5 * (P_np + P_np.T)   # qpsolvers recommendation
q_np  = -as_vec(q)
G_np  = as_mat(GhAb['G'])
h_np  = as_vec(GhAb['h'])
A_np  = as_mat(GhAb['A']) if GhAb['A'] is not None else None
b_np  = as_vec(GhAb['b']) if GhAb['b'] is not None else None
lb_np = as_vec(constraints.box['lower'])
ub_np = as_vec(constraints.box['upper'])



# 5. Loop over all available solvers, solve the problem, and store the results
for s in available_solvers:
    try:

        # Create a QuadraticProgram instance with the problem data and the current solver.
        qp = QuadraticProgram(
            P=P_np * risk_aversion,
            q=q_np,
            G=G_np,
            h=h_np,
            A=A_np,
            b=b_np,
            lb=lb_np,
            ub=ub_np,
            solver=s,
        )

        # Measure the runtime of the solver
        t0 = time.perf_counter()
        qp.solve()
        runtime = time.perf_counter() - t0

        # Extract the solution
        solution = qp.results.get('solution')
        
        # Check if a solution was returned and store the results accordingly
        if solution is not None:
            # try to recover portfolio weights
            weights = qp.results.get("x", None)
            if weights is None:
                weights = getattr(solution, "x", None)
            
            # store diagnostics
            results_dict[s] = {
                "solution_found": solution.found,
                "objective": qp.objective_value(),
                "primal_residual": solution.primal_residual(),
                "dual_residual": solution.dual_residual(),
                "duality_gap": solution.duality_gap(),   # no [0]
                "runtime": runtime,
                "error": None,
            }
            
            # store weights separately
            if weights is not None:
                weights_dict[s] = pd.Series(weights, index=constraints.ids)
            else:
                weights_dict[s] = None

        else:
            results_dict[s] = {
                **result_on_fail,
                "error": "No solution returned",
            }
            weights_dict[s] = None
            print(f"Solver '{s}' error: No solution returned")
            

    except Exception as e:
        results_dict[s] = {
            **result_on_fail,
            "error": str(e),
        }
        weights_dict[s] = None
        print(f"Solver '{s}' error: {e}")



# 6. Convert the results dictionary to a DataFrame for easy comparison
datafr = pd.DataFrame(results_dict)

Print and visualize the results

In [7]:
# 7. Print the results DataFrame with solvers as columns and metrics as rows
display(datafr)

,daqp,qpalm,osqp,quadprog,cvxopt
solution_found,True,True,True,True,True
objective,-0.000778,-0.000778,-0.00081,-0.000778,-0.000778
primal_residual,0.0,0.000001,0.001023,0.0,0.0
dual_residual,0.0,0.000072,0.000002,0.0,0.0
duality_gap,0.0,0.000012,0.000032,0.0,0.0
runtime,0.000891,0.005913,0.003923,0.000417,0.048399
error,None,None,None,None,None


## 2. Analytical Solution to Minimum-Variance Problem

(5 points)

- Create a `MinVariance` class that follows the structure of the `MeanVariance` class.
- Implement the `solve` method in `MinVariance` such that if `solver_name = 'analytical'`, the analytical solution is computed and stored within the object (if such a solution exists). If not, call the `solve` method from the parent class.
- Create a `Constraints` object by injecting the same ids as in part 1.b) and add a budget constraint.
- Instantiate a `MinVariance` object by setting `solver_name = 'analytical'` and passing instances of `Constraints` and `Covariance` as arguments.
- Create an `OptimizationData` object that contains an element `return_series`, which consists of the synthetic data generated in part 1.a).
- Solve the optimization problem using the created `MinVariance` object and compare the results to those obtained in part 1.c).


In [8]:
# Define class MinVariance
class MinVariance(Optimization):
    def __init__(self,
                 constraints: Constraints,
                 covariance: Optional[Covariance] = None,
                 **kwargs):
        super().__init__(constraints=constraints, **kwargs)
        self.covariance = Covariance() if covariance is None else covariance

    def set_objective(self, optimization_data: OptimizationData) -> None:
        X = optimization_data['return_series']
        Sigma = self.covariance.estimate(X=X, inplace=False)
        n_assets = X.shape[1]

        self.objective = Objective(
            q = np.zeros(n_assets),
            P = 2* Sigma,
        )
        return None

    # ==============================
    # Helper Functions for Analytical Solution
    # ==============================

    @staticmethod
    def _to_numpy(x):
        """Convert pandas / array-like objects to NumPy arrays."""
        if hasattr(x, "to_numpy"):
            x = x.to_numpy()
        return np.asarray(x, dtype=float)
    
    # Helper function to extract Sigma from the quadratic objective
    def _extract_sigma(self) -> np.ndarray:
        """Recover Sigma from the quadratic objective P = 2 * Sigma."""
        if self.objective is None:
            raise ValueError("Objective is not set. Call set_objective() first.")

        coeffs = self.objective.coefficients
        if "P" not in coeffs:
            raise ValueError("Objective must contain 'P'.")

        P = self._to_numpy(coeffs["P"])
        Sigma = 0.5 * P
        return Sigma

    # Helper function to extract equality constraints A and b
    def _extract_equality_constraints(self) -> tuple[np.ndarray, np.ndarray]:
        """
        Extract equality constraints A x = b.

        The analytical minimum-variance formula is valid only for
        equality-constrained problems.
        """
        GhAb = self.constraints.to_GhAb()
        A = GhAb.get("A", None)
        b = GhAb.get("b", None)
        G = GhAb.get("G", None)

        if A is None or b is None:
            raise ValueError(
                "Analytical solution requires equality constraints A x = b."
            )

        if G is not None and len(G) > 0:
            raise ValueError(
                "Analytical solution is only implemented for equality constraints."
            )

        A = self._to_numpy(A)
        b = self._to_numpy(b).reshape(-1, 1)
        return A, b
    
    # ==============================
    # Analytical Solution Function
    # ==============================
    
    def _solve_analytically(self) -> None:
        """
        Solve the minimum-variance problem analytically using:

            x* = Σ^{-1} A' (A Σ^{-1} A')^{-1} b
        """
        Sigma = self._extract_sigma()
        A, b = self._extract_equality_constraints()

        # Step 1: compute Σ^{-1} A'
        Sigma_inv_AT = np.linalg.solve(Sigma, A.T)

        # Step 2: compute A Σ^{-1} A'
        middle_matrix = A @ Sigma_inv_AT

        # Step 3: compute (A Σ^{-1} A')^{-1} b
        multiplier = np.linalg.solve(middle_matrix, b)

        # Step 4: compute x*
        x = Sigma_inv_AT @ multiplier
        x = x.reshape(-1)

        # Store results
        self.results["weights"] = x
        self.results["objective_value"] = float(x @ Sigma @ x)
    
    # ==============================
    # Optimizer Solve
    # ==============================

    def solve(self) -> None:
        """
        Solve the optimization problem.

        - If solver_name == 'analytical', use the closed-form solution.
        - Otherwise, call the parent class solver.
        """
        if self.params.get("solver_name") != "analytical":
            return super().solve()

        self._solve_analytically()
        return None


In [13]:
# Create a constraints object with just a budget constraint
constraints = Constraints(ids=returns.columns.tolist())
constraints.add_budget(rhs=1, sense="=")

covariance = Covariance(method='pearson')
covariance.estimate(X=returns, inplace=True)

# Instantiate the MinVariance class
minvar = MinVariance(
    constraints=constraints,
    covariance=covariance,
    solver_name="analytical",
)

# Prepare the optimization data and prepare the optimization problem
optimization_data = OptimizationData(sigma=covariance.matrix)
optimization_data["return_series"] = returns

# Solve the optimization problem and print the weights
minvar.set_objective(optimization_data=optimization_data)
minvar.objective.coefficients

minvar.solve()
print(minvar.results)

{'weights': array([0.01710024, 0.01884803, 0.01799181, 0.01694421, 0.01989986,
       0.01878412, 0.02111555, 0.01833781, 0.01773845, 0.02520369,
       0.02014201, 0.01171339, 0.01509241, 0.01287224, 0.01952433,
       0.02171886, 0.01727802, 0.01854871, 0.02539835, 0.01497258,
       0.01265498, 0.0283269 , 0.02159265, 0.02556342, 0.01825449,
       0.01729261, 0.01703572, 0.01846931, 0.02052469, 0.01603572,
       0.01930438, 0.02309176, 0.01754649, 0.01721165, 0.01841837,
       0.0222425 , 0.02734901, 0.01345428, 0.02918804, 0.02542498,
       0.01773154, 0.02275299, 0.02115352, 0.02544577, 0.02095374,
       0.01976864, 0.01642057, 0.02684582, 0.02803725, 0.02268353]), 'objective_value': 2.0317760354452115e-05}
